# Chapter 8 &mdash; Error-Correcting Design I: the RE for Hamming Distance 2

**Concept 6 of the Chapter 8 decomposition:** *Error-Correcting Design I: the RE for "within Hamming Distance 2"*

Enumerate the $\binom{4}{2}=6$ ways of denting `0101`, write `?` as `(0+1)`, and union them.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-Hamming-Distance-RE/Concept-Hamming-Distance-RE.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Design task: accept every 4-bit string **within Hamming distance 2** of `0101`.

The systematic construction: a string at distance **exactly** 2 differs in exactly two
positions, and there are $\binom{4}{2}=6$ choices of which two. For each choice write
the pattern with those positions **denteded** &mdash; a "don't care", written `(0+1)`.

Union the six, and add the distance-1 and distance-0 cases &mdash; or note that
`(0+1)` covers the original symbol too, so the six patterns already include distances
0, 1 and 2.

The count is a sanity check you can compute independently:
$\sum_{k\le2}\binom{4}{k} = 1+4+6 = 11$ strings.

## 2. Definitions

### Build the RE from the position choices

In [ ]:
from itertools import combinations, product
TARGET = '0101'

def dent_re(target, k):
    alts = []
    for pos in combinations(range(len(target)), k):
        alts.append(''.join('(0+1)' if i in pos else target[i]
                            for i in range(len(target))))
    return alts

alts = dent_re(TARGET, 2)
RE2 = '+'.join(alts)
print("%d alternatives, e.g. %s" % (len(alts), alts[0]))

### The reference: Hamming distance

In [ ]:
def ham(a, b): return sum(x != y for x, y in zip(a, b))
def within2(s): return len(s) == len(TARGET) and ham(s, TARGET) <= 2

## 3. Tests

Six alternatives, as $\binom{4}{2}$ predicts.

In [ ]:
from math import comb
print("C(4,2) =", comb(4, 2), " alternatives built :", len(alts))
assert len(alts) == comb(4, 2)

The RE accepts exactly the strings within distance 2.

In [ ]:
def re_dfa(r): return min_dfa(nfa2dfa(re2nfa(r)))
D = re_dfa(RE2)
acc = [''.join(p) for p in product('01', repeat=4) if accepts_dfa(D, ''.join(p))]
print("accepted (%d) :" % len(acc), acc)
assert set(acc) == {''.join(p) for p in product('01', repeat=4)
                    if within2(''.join(p))}

The count matches $\sum_{k\le2}\binom{4}{k} = 11$.

In [ ]:
print("1 + 4 + 6 =", comb(4,0) + comb(4,1) + comb(4,2))
assert len(acc) == comb(4,0) + comb(4,1) + comb(4,2) == 11

Distance 0, 1 and 2 are all present &mdash; the dents subsume the smaller distances.

In [ ]:
from collections import Counter
print("by distance :", dict(sorted(Counter(ham(s, TARGET) for s in acc).items())))
assert TARGET in acc, "distance 0 must be included"
assert max(ham(s, TARGET) for s in acc) == 2

Nothing at distance 3 or 4 sneaks in.

In [ ]:
outside = [''.join(p) for p in product('01', repeat=4)
           if ham(''.join(p), TARGET) > 2]
assert not any(accepts_dfa(D, s) for s in outside)
print("all %d strings at distance 3 or 4 are rejected" % len(outside))

## 4. Animation

The distance-2 acceptor for `0101`.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(re_dfa(RE2), FuseEdges=True)

## 5. Exercises


1. Build the distance-1 RE. How many alternatives, and how many strings?
2. Generalise `dent_re` to distance $k$ over a 5-bit target. Check the count.
3. Why does `(0+1)` in a dent position also cover the *original* symbol?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter8/Concept-Hamming-Distance-RE')